In [35]:
import pandas as pd

USERS

In [36]:
users = pd.read_csv(r"C:\Users\ASUS\Downloads\Users.csv")

In [37]:
users.head()

,User_id,Name,Age,Gender,Marital Status,Occupation
0,1,Claire Ferguson,20,Female,Single,Student
1,2,Jennifer Young,24,Female,Single,Student
2,3,Jermaine Roberson,22,Male,Single,Student
3,4,Rachel Carpenter,22,Female,Single,Student
4,5,Shawn Parker,22,Male,Single,Student


In [38]:
users.dtypes

User_id            int64
Name              object
Age                int64
Gender            object
Marital Status    object
Occupation        object
dtype: object

In [39]:
users.columns = (users.columns.str.strip().str.lower().str.replace(" ", "_"))
users.columns

Index(['user_id', 'name', 'age', 'gender', 'marital_status', 'occupation'], dtype='object')

In [40]:
print(users["gender"].value_counts())
print(users["marital_status"].value_counts())
print(users["occupation"].value_counts())

gender
Male      57218
Female    42782
Name: count, dtype: int64
marital_status
Single               69077
Married              27831
Prefer not to say     3092
Name: count, dtype: int64
occupation
Student           53357
Employee          30408
Self Employeed    13913
House wife         2322
Name: count, dtype: int64


In [41]:
text_columns = ["name", "gender", "marital_status", "occupation"]

for col in text_columns:
    users[col] = users[col].str.strip()

In [42]:
print("Shape:", users.shape)
print("Missing values:")
print(users.isnull().sum())

print("Duplicate rows:", users.duplicated().sum())

Shape: (100000, 6)
Missing values:
user_id           0
name              0
age               0
gender            0
marital_status    0
occupation        0
dtype: int64
Duplicate rows: 0


In [43]:
users.to_csv("../data/processed/users_cleaned.csv", index=False)

ORDERS

In [44]:
orders = pd.read_csv(r"C:\Users\ASUS\Downloads\Orders.csv")

In [45]:
orders.head()

,order_date,Sales_QTY,Sales_amount,currency,User_id,Restaurant_id
0,10/10/2017,100,41241,INR,49226,567335.0
1,5/8/2018,3,-1,INR,77359,531342.0
2,4/6/2018,1,875,INR,5321,158203.0
3,4/11/2018,1,583,INR,21343,187912.0
4,6/18/2018,6,7176,INR,75378,543530.0


In [46]:
orders.columns = (orders.columns.str.strip().str.lower().str.replace(" ", "_"))
orders.columns

Index(['order_date', 'sales_qty', 'sales_amount', 'currency', 'user_id',
       'restaurant_id'],
      dtype='object')

CONVERT ORDER_DATE TO DATETIME

In [47]:
orders["order_date"] = pd.to_datetime(orders["order_date"],errors="coerce")

In [48]:
orders["order_date"].dtype

dtype('<M8[ns]')

In [49]:
orders.isnull().sum()

order_date          0
sales_qty           0
sales_amount        0
currency            0
user_id             0
restaurant_id    1617
dtype: int64

In [50]:
# CHECK INVALID SALES AMOUNTS
orders[orders["sales_amount"] <= 0][["order_date", "sales_qty", "sales_amount", "user_id", "restaurant_id"]].head(20)

,order_date,sales_qty,sales_amount,user_id,restaurant_id
1,2018-05-08,3,-1,77359,531342.0
159,2018-05-26,1,0,59836,102582.0
196,2019-04-30,1,0,95267,553404.0
239,2018-12-28,1,0,14159,499377.0
282,2018-05-08,3,-1,357,509389.0
438,2018-05-26,1,0,38713,389125.0
475,2019-04-30,1,0,33710,544517.0
518,2018-12-28,1,0,3624,156634.0
583,2018-01-10,5,0,85273,400695.0
1541,2017-10-10,1,0,82531,84145.0


In [51]:
print("Negative sales:", (orders["sales_amount"] < 0).sum())
print("Zero sales:", (orders["sales_amount"] == 0).sum())

Negative sales: 2
Zero sales: 1609


In [52]:
orders = orders[orders["sales_amount"] > 0].copy()

In [53]:
# Check very large quantities

orders["sales_qty"].describe()

count    148670.000000
mean         16.355277
std         115.295762
min           1.000000
25%           1.000000
50%           1.000000
75%           7.000000
max       14049.000000
Name: sales_qty, dtype: float64

In [54]:
orders.sort_values("sales_qty",ascending=False)[["order_date", "sales_qty", "sales_amount", "user_id", "restaurant_id"]].head(20)

,order_date,sales_qty,sales_amount,user_id,restaurant_id
22034,2019-11-29,14049,628657,60046,416409.0
21547,2018-02-08,10667,450954,58535,385543.0
21805,2019-01-25,10667,534463,78855,407770.0
21663,2018-07-23,10667,534463,43819,537634.0
63986,2018-01-31,8800,647778,66926,542825.0
21313,2020-02-29,8000,38889,97856,286268.0
21990,2019-09-30,6667,298310,71745,493031.0
21314,2020-04-14,6000,29167,73359,230460.0
21475,2017-11-10,5333,225477,11486,493145.0
21779,2018-12-14,5333,267231,14698,532375.0


In [55]:
orders["sales_qty"].value_counts().head(20)

sales_qty
1     80397
2     11561
3      7613
7      4724
4      3888
5      3308
6      2655
13     2458
8      2025
12     1640
10     1591
40     1430
11     1312
16     1274
9      1005
53      969
20      867
23      820
21      801
24      743
Name: count, dtype: int64

In [56]:
orders[orders["sales_qty"] > 1000][["sales_qty", "sales_amount"]].describe()

,sales_qty,sales_amount
count,142.000000,1.420000e+02
mean,2656.147887,1.539160e+05
std,2155.826938,2.162190e+05
min,1049.000000,1.296300e+04
25%,1333.000000,6.470150e+04
50%,1695.500000,8.422000e+04
75%,3299.750000,1.331320e+05
max,14049.000000,1.338264e+06


In [57]:
orders["sales_qty"].corr(orders["sales_amount"])

np.float64(0.3441587416005891)

In [58]:
orders["currency"].value_counts()

currency
INR    148670
Name: count, dtype: int64

In [59]:
orders = orders[orders["sales_qty"] <= 1000].copy()

In [60]:
print("Remaining rows:", len(orders))
print("Orders with sales_qty > 1000:",
      (orders["sales_qty"] > 1000).sum())

Remaining rows: 148528
Orders with sales_qty > 1000: 0


In [61]:
orders["sales_qty"].describe()

count    148528.000000
mean         13.831507
std          47.171140
min           1.000000
25%           1.000000
50%           1.000000
75%           7.000000
max        1000.000000
Name: sales_qty, dtype: float64

In [62]:
orders["sales_qty"].value_counts().head(15)

sales_qty
1     80397
2     11561
3      7613
7      4724
4      3888
5      3308
6      2655
13     2458
8      2025
12     1640
10     1591
40     1430
11     1312
16     1274
9      1005
Name: count, dtype: int64

In [63]:
print("Invalid dates:", orders["order_date"].isna().sum())

Invalid dates: 0


In [64]:
orders.to_csv("../data/processed/orders_cleaned.csv",index=False)

In [65]:
orders.shape

(148528, 6)

In [66]:
# Check unmatched Restaurant IDs

restaurants = pd.read_csv(r"C:\Users\ASUS\Downloads\Restaurant.csv")

In [67]:
valid_restaurant_ids = set(restaurants["Restaurant_id"])

unmatched = orders[orders["restaurant_id"].notna() &
    ~orders["restaurant_id"].isin(valid_restaurant_ids)
]

print("Unmatched restaurant IDs:", len(unmatched))


Unmatched restaurant IDs: 85


RESTAURANTS

In [68]:
restaurants = pd.read_csv(r"C:\Users\ASUS\Downloads\Restaurant.csv")

In [69]:
restaurants.head()

,Restaurant_id,Name,Country,City,Rating,Rating_count,Cuisine
0,567335,AB FOODS POINT,India,Abohar,--,Too Few Ratings,"Beverages,Pizzas"
1,531342,Janta Sweet House,India,Abohar,4.4,50+ ratings,"Sweets,Bakery"
2,158203,theka coffee desi,India,Abohar,3.8,100+ ratings,Beverages
3,187912,Singh Hut,India,Abohar,3.7,20+ ratings,"Fast Food,Indian"
4,543530,GRILL MASTERS,India,Abohar,--,Too Few Ratings,"Italian-American,Fast Food"


In [70]:
#Standardize column names

restaurants.columns = ( restaurants.columns.str.strip().str.lower().str.replace(" ","_"))
restaurants.columns

Index(['restaurant_id', 'name', 'country', 'city', 'rating', 'rating_count',
       'cuisine'],
      dtype='object')

In [71]:
#Remove extra spaces from text columns

text_columns = ["name","country","city","rating_count","cuisine"]

for col in text_columns:
    restaurants[col] = restaurants[col].str.strip()

In [72]:
# Convert valid ratings to numbers:

restaurants["rating"] = pd.to_numeric(restaurants["rating"],errors = "coerce")


print(restaurants["rating"].dtype)
print(restaurants["rating"].isna().sum())

float64
87013


In [73]:
print(restaurants.loc[restaurants["rating"].notna(),"rating"].unique()[:20])

[4.4 3.8 3.7 3.6 4.  4.2 4.7 4.1 3.5 3.1 3.4 3.3 4.8 3.9 2.7 4.3 2.9 4.5
 2.5 3.2]


In [74]:
restaurants["rating"].isnull().sum()

np.int64(87013)

In [75]:
restaurants["rating"].describe()

count    61441.000000
mean         3.894461
std          0.460058
min          1.000000
25%          3.700000
50%          4.000000
75%          4.200000
max          5.000000
Name: rating, dtype: float64

In [76]:
restaurants["rating"].value_counts().sort_index().head(20)

rating
1.0      1
1.1      1
1.2      4
1.3      3
1.4     10
1.5     18
1.6     17
1.7     13
1.8     27
1.9     46
2.0     72
2.1     61
2.2     89
2.3    128
2.4    118
2.5    221
2.6    208
2.7    325
2.8    473
2.9    472
Name: count, dtype: int64

In [77]:
restaurants["rating_count_numeric"] = pd.to_numeric(
    restaurants["rating_count"],
    errors="coerce"
)

In [78]:
print(restaurants[["rating_count", "rating_count_numeric"]].head())
print("Missing numeric values:", restaurants["rating_count_numeric"].isna().sum())

      rating_count  rating_count_numeric
0  Too Few Ratings                   NaN
1      50+ ratings                   NaN
2     100+ ratings                   NaN
3      20+ ratings                   NaN
4  Too Few Ratings                   NaN
Missing numeric values: 148454


In [79]:
print(restaurants[["rating", "rating_count_numeric"]].head(10))
print(restaurants["rating_count_numeric"].dtype)

   rating  rating_count_numeric
0     NaN                   NaN
1     4.4                   NaN
2     3.8                   NaN
3     3.7                   NaN
4     NaN                   NaN
5     3.6                   NaN
6     4.0                   NaN
7     NaN                   NaN
8     4.2                   NaN
9     NaN                   NaN
float64


In [80]:
restaurants["rating_count"].value_counts().head(30)

rating_count
Too Few Ratings    87013
20+ ratings        21636
100+ ratings       20548
50+ ratings        12009
500+ ratings        4396
1K+ ratings         2739
5K+ ratings           98
10K+ ratings          15
Name: count, dtype: int64

In [81]:
def convert_rating_count(value):
    if value == "Too Few Ratings":
        return None
    
    value = value.replace("+ ratings", "").strip()
    
    if "K" in value:
        return float(value.replace("K", "")) * 1000
    
    return float(value)


restaurants["rating_count_numeric"] = (
    restaurants["rating_count"]
    .apply(convert_rating_count)
)

In [82]:
restaurants[["rating_count", "rating_count_numeric"]].drop_duplicates().sort_values("rating_count_numeric")

,rating_count,rating_count_numeric
3,20+ ratings,20.0
1,50+ ratings,50.0
2,100+ ratings,100.0
88,500+ ratings,500.0
133,1K+ ratings,1000.0
6489,5K+ ratings,5000.0
71678,10K+ ratings,10000.0
0,Too Few Ratings,NaN


In [83]:
text_columns = ["name","country","city","rating_count","cuisine"]

for col in text_columns:
    restaurants[col] = restaurants[col].str.strip()

In [84]:
restaurants.isnull().sum()

restaurant_id               0
name                        0
country                     0
city                        0
rating                  87013
rating_count                0
cuisine                     0
rating_count_numeric    87013
dtype: int64

In [85]:
print("Duplicate rows:", restaurants.duplicated().sum())

Duplicate rows: 0


In [86]:
restaurants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148454 entries, 0 to 148453
Data columns (total 8 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   restaurant_id         148454 non-null  int64  
 1   name                  148454 non-null  object 
 2   country               148454 non-null  object 
 3   city                  148454 non-null  object 
 4   rating                61441 non-null   float64
 5   rating_count          148454 non-null  object 
 6   cuisine               148454 non-null  object 
 7   rating_count_numeric  61441 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 9.1+ MB


In [87]:
restaurants.to_csv(
    "../data/processed/restaurants_mysql.csv",
    index=False,
    na_rep=""
)

In [88]:
with open(
    "../data/processed/restaurants_mysql.csv",
    "r",
    encoding="utf-8"
) as f:
    for i in range(5):
        print(f.readline().strip())

restaurant_id,name,country,city,rating,rating_count,cuisine,rating_count_numeric
567335,AB FOODS POINT,India,Abohar,,Too Few Ratings,"Beverages,Pizzas",
531342,Janta Sweet House,India,Abohar,4.4,50+ ratings,"Sweets,Bakery",50.0
158203,theka coffee desi,India,Abohar,3.8,100+ ratings,Beverages,100.0
187912,Singh Hut,India,Abohar,3.7,20+ ratings,"Fast Food,Indian",20.0


In [89]:
print(restaurants.columns.tolist())

['restaurant_id', 'name', 'country', 'city', 'rating', 'rating_count', 'cuisine', 'rating_count_numeric']
